# 1. Import
## 1.1. FastSAM Module

In [1]:
import os, sys

sys.path.insert(0, r'D:\01_Floor\a_Ed\09_EECS\10_Python\90_Online Tools\FastSAM')
from fastsam import FastSAM, FastSAMPrompt #type: ignore

#Define directory for FastSAM model checkpoint
path_FastSAM_model_pt = r'D:\01_Floor\a_Ed\09_EECS\10_Python\weights\FastSAM-x.pt'
model = FastSAM(path_FastSAM_model_pt)
DEVICE = 'cpu' #Change to gpu is applicable

## 1.2. Functions

In [2]:
import re, cv2
import numpy as np
import pandas as pd
from PIL import Image
import math
import torch #type: ignore
np.set_printoptions(suppress=True)

# Enhance contrast function
def enhance(image_path):
  img = cv2.imread(image_path, 1)
  #[1] converting to LAB color space
  lab= cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
  l_channel, a, b = cv2.split(lab)

  #[2] Applying CLAHE to L-channel
  clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
  cl = clahe.apply(l_channel)

  #[3] merge the CLAHE enhanced L-channel with the a and b channel
  limg = cv2.merge((cl,a,b))

  #[4] Converting image from LAB Color model to BGR color spcae
  enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
  path_enhanced_img = image_path.replace('.png','_enhanced.png')
  cv2.imwrite(path_enhanced_img, enhanced_img)

def find_extreme_values(ann_np, dep):
  #[1] Find the top most non-zero element
  r_up = np.argmax(np.sum(ann_np[dep], axis=1) > 0)
  c_up = np.argmax(ann_np[dep, r_up] > 0)
  up = [r_up, c_up]

  #[2] Find the left most non-zero element
  c_left = np.argmax(np.sum(ann_np[dep], axis=0) > 0)
  r_left = np.argmax(ann_np[dep, :, c_left] > 0)
  left = (r_left, c_left)

  #[3] Find the bottom most non-zero element
  row_sums = np.sum(ann_np[dep], axis=1)
  r_down = np.max(np.where(row_sums > 0, np.arange(row_sums.size), -1))
  c_down = np.argmax(ann_np[dep, r_down] > 0)
  down = [r_down, c_down]

  #[4] Find the right most non-zero element
  c_right = ann_np.shape[2] - np.argmax(np.flip(np.sum(ann_np[dep], axis=0), axis=0) > 0) - 1
  r_right = np.max(np.where(ann_np[dep, :, c_right] > 0, np.arange(ann_np.shape[1]), -1))
  right = [r_right, c_right]

  return up, left, down, right

def find_indicies(up, down, left, right):
  point_1 = [0,0]
  point_2 = [0,0]
  point_3 = [0,0]
  
  #[1] top = left
  if abs(up[1] - left[1]) < 25:
    point_1 = [up[0],left[1]]
    point_2 = [down[0],down[1]]
    point_3 = [right[0],right[1]]
  #[2] top = right
  elif abs(up[1] - right[1]) < 25:
    point_1 = [up[0],right[1]]
    point_2 = [down[0],down[1]]
    point_3 = [left[0],left[1]]
  #[3] bottom = left
  elif abs(down[1] - left[1]) < 25:
    point_1 = [down[0],left[1]]
    point_2 = [up[0],up[1]]
    point_3 = [right[0],right[1]]
  #[4] bottom = right
  elif abs(down[1] - right[1]) < 25:
    point_1 = [down[0],right[1]]
    point_2 = [up[0],up[1]]
    point_3 = [left[0],left[1]]
     
  pos_1.append(point_1)
  pos_2.append(point_2)
  pos_3.append(point_3)
  return  point_1, point_2, point_3, pos_1, pos_2, pos_3

def calculate_parameters(point_1, point_2, point_3):
  
  #[1] Find area using vertex
  x1 = point_1[0]
  x2 = point_2[0]
  x3 = point_3[0]
  y1 = point_1[1]
  y2 = point_2[1]
  y3 = point_3[1]
  area_from_pt = abs(0.5*(x1*(y2-y3)+x2*(y3-y1)+x3*(y1-y2)))
  error = round(100*(area_from_pt-count)/count,2)
  
  #[2] Find area from boxing
  area_from_box = 0.5*(max(x1,x2,x3)-min(x1,x2,x3))*(max(y1,y2,y3)-min(y1,y2,y3))
  error_box = round(100*(area_from_box-count)/count,2)
  
  calc_area.append(area_from_pt)
  errors.append(error)
  area_box.append(area_from_box)
  errors_box.append(error_box)
    
  #[3] Calculate side length
  length_1 = math.dist(point_1,point_2)
  length_2 = math.dist(point_1,point_3)
  length_3 = math.dist(point_2,point_3)
  delta = round(max(length_1,length_2,length_3) - min(length_1,length_2,length_3),2)
  average = round((length_1 + length_2 + length_3)/3,2)
  lengths_1.append(length_1)
  lengths_2.append(length_2)
  lengths_3.append(length_3)
  deltas.append(delta)
  averages.append(average)
  return calc_area, area_box, errors, errors_box, lengths_1, lengths_2, lengths_3, deltas, averages


# 2. Processing

In [3]:
#[1] Image
#[5] Define input and output directory
folder_in = r'D:\01_Floor\a_Ed\09_EECS\10_Python\03_MatureTools\2024-0828_SEM image segmentation\SIS-04_Input'
folder_out = r'D:\01_Floor\a_Ed\09_EECS\10_Python\03_MatureTools\2024-0828_SEM image segmentation\SIS-05_Output'


image_list = os.listdir(folder_in)
image_list = [file for file in image_list if os.path.isfile(os.path.join(folder_in, file)) 
              and not 'reduced' in file and not 'corrected' in file and not 'sobelxy' in file
              and not 'Blur' in file and not 'grayScaled' in file and not 'binary' in file
              and not 'line' in file and not 'test' in file and not 'enhanced' in file]


"""#[6] Remove .DS_Store file created by MacOS
if '.DS_Store' in image_list:
    image_list.remove('.DS_Store')"""

#[8] Enhance contrast of Image
for x in image_list:
  image_path = folder_in + x
  enhance(image_path)

for idx, img_name in enumerate(image_list):
  #Use enhanced images for segmentation
  x = re.sub('\.png', '_enhanced.png',img_name)
  path = os.path.join(folder_in, x)
  
  #Image Segmentation
  everything_results = model(path, device=DEVICE, retina_masks=True, imgsz=1024, conf=0.4, iou=0.9)
  prompt_process = FastSAMPrompt(path, everything_results, device=DEVICE)
  ann = prompt_process.everything_prompt()
  
  #Covert tensor into np array
  ann_np = ann.numpy()
  depth,row,column = ann_np.shape
  
  #check for crystalson the edge of the image and remove them
  edge_pixels = []
  layers_to_remove = []
  for dep in range (depth):
    edge_index = (np.sum(ann_np[dep,0,:]) + np.sum(ann_np[dep,row-1,:]) + np.sum(ann_np[dep,:,0]) + np.sum(ann_np[dep,:,column-1]))
    edge_pixels.append(edge_index)
    
    #Remove masks with more than 20 pixels on the edge and with area < 2000 pixels
    if edge_index > 20 or torch.sum(ann[dep]).item() < 2000: #Change threshold if necessary
        ann_np[dep,:,:] = 0
    if np.all(ann_np[dep] == 0):
        layers_to_remove.append(dep)
  ann_np = np.delete(ann_np, layers_to_remove, axis=0)
  ann = torch.from_numpy(ann_np)

  # Initiallize parameters
  areas = []
  pos_1 = []
  pos_2 = []
  pos_3 = []
  calc_area = []
  area_box = []
  errors = [] #%Error from calculating area with three indicies
  errors_box = [] #%Error from calculating 0.5 times the box enclosing the shape
  lengths_1 = []
  lengths_2 = []
  lengths_3 = []
  deltas = [] #Range of lengths
  averages = [] #Average of lengths
  
  depth,row,column = ann_np.shape
  for dep in range(depth):
    count = torch.sum(ann[dep]).item()
    areas.append(count)
  
    #Find the 4 extreme values of each shape
    up, left, down, right = find_extreme_values(ann_np, dep)

    #Find the indecies of triangle
    point_1, point_2, point_3, pos_1, pos_2, pos_3 = find_indicies(up, down, left, right)

    # Calculate all parameters
    calc_area, area_box, errors, errors_box, lengths_1, lengths_2, lengths_3, deltas, averages = calculate_parameters(point_1, point_2, point_3) 

  #Define output names
  y = re.sub('\.png', '',image_list[idx])
  output_filename = y+'_Output.png'
  output_path = os.path.join(folder_out, output_filename)
  
  #Save output parameters
  data = {
    "pos 1": pos_1,
    "pos 2": pos_2,
    "pos 3": pos_3,
    "Actual Area": areas,
    "Area (Verticies)": calc_area,
    "%Error (Verticies)": errors,
    "Area (Box)": area_box,
    "%Error (Box)": errors_box,
    "Side 1": lengths_1,
    "Side 2": lengths_2,
    "Side 3": lengths_3,
    "Range": deltas,
    "Average length": averages
    }
  df = pd.DataFrame(data)
  excel_file_path = folder_out + y + '_analysis.xlsx'
  df.to_excel(excel_file_path, index=False)

  #Save output images 
  prompt_process.plot(annotations=ann_np, withContours=False, output_path=output_path)

error: OpenCV(4.8.1) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
